In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import RandomOverSampler, SMOTE

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder, StandardScaler

In [ ]:
df = pd.read_csv("/content/insurance_claims.csv")
df.head()

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,...,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,...,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,...,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,...,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,...,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


In [ ]:
df.drop(columns="policy_bind_date" , inplace=True)

# **Finding the Outliers**

In [ ]:
df.shape

(1000, 39)

In [ ]:
num_cols = []

for col in df.columns:
    if (df[col].dtype =='int'):
        num_cols.append(col)

In [ ]:
len(num_cols)

17

In [ ]:
def remove_outliers(data):
  data_set = data.copy()
  list_of_set_data = list()


  for col in df[num_cols]:
    # 25 percentile
    q1 = data_set[col].quantile(0.25)
    # 75 percentile
    q3 = data_set[col].quantile(0.75)
    # Interquartile Range
    iqr = q3 - q1

    data_cleaned = data_set[~((data_set[col] < (q1-1.5*iqr))|
                              (data_set[col] > (q3-1.5*iqr))
                              )].copy()

    list_of_set_data.append(data_cleaned.copy())



  data_cleaned = pd.concat(list_of_set_data)
  '''
  count_duplicated_index = data_cleaned.index.value_counts()
  used_index_data = count_duplicated_index[count_duplicated_index == len(df['int32_col'])].index
  data_cleaned = data_cleaned.loc[used_index_data].drop_duplicates()

  '''
  return data_cleaned





In [ ]:
df = remove_outliers(df)

# **Handling Misiing Values**

In [ ]:
# Total Number of missing values in all numeric columns
df[num_cols].isnull().sum()

,0
months_as_customer,0
age,0
policy_number,0
policy_deductable,0
umbrella_limit,0
insured_zip,0
capital-gains,0
capital-loss,0
incident_hour_of_the_day,0
number_of_vehicles_involved,0


In [ ]:
# Total Number of missing values in Categorical columns
cat_cols = [col for col in df.columns if df[col].dtype == "object"]
# df[cat_cols].isnull().sum()
df[cat_cols].shape

(2017, 20)


**Handling missing value on numeric features**

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

def imputer_Num(data, imputer=None):

    data.drop(columns = "_c39" , inplace = True)

    num_cols = [col for col in data.columns if data[col].dtype in ['int64', 'int32', 'float64']]

    if imputer is None:
        imputer = SimpleImputer(missing_values=np.nan, strategy="median")
        imputer.fit(data[num_cols])

    data_imputed = pd.DataFrame(
        imputer.transform(data[num_cols]),
        index=data.index,
        columns=num_cols
    )

    return data_imputed


In [ ]:
df_numeric = imputer_Num(df)


**Handling missing value on categorical feature**

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

def imputer_Cat(data, imputer=None):

    data = data.replace('?', np.nan)

    # data.drop(columns = "fraud_reported" , inplace = True)

    cat_cols = [col for col in data.columns if data[col].dtype == "object"]

    if imputer is None:
        imputer = SimpleImputer(missing_values=np.nan, strategy="constant" , fill_value="UNKNOWN")

        imputer.fit(data[cat_cols])

    data_imputed = pd.DataFrame(
        imputer.transform(data[cat_cols]),
        index=data.index,
        columns=cat_cols
    )

    return data_imputed


In [ ]:
df_categorical = imputer_Cat(df)

In [ ]:
# Concatenating Both numeric and categorical feature after applying imputer
df = pd.concat(
    [df_numeric, df_categorical],
    axis=1
)


# **Scaling Data**

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def scaling_data(data, scaler=None):


    num_cols = data.select_dtypes(include=['int64', 'int32', 'float64']).columns.tolist()


    if len(num_cols) == 0:
        raise ValueError("No numeric columns found for scaling")

    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(data[num_cols])

    data_scaled = pd.DataFrame(
        scaler.transform(data[num_cols]),
        index=data[num_cols].index,
        columns=num_cols
    )

    return data_scaled, scaler


In [ ]:
df_scaled, scaler = scaling_data(df)

df = pd.concat(
    [df.drop(columns=df_scaled.columns), df_scaled],
    axis=1
)



# **Encoding Teachnique**

In [ ]:
cat_cols = [col for col in df.columns if df[col].dtype == "object"]


In [ ]:
nominal = ['authorities_contacted', 'incident_state', 'insured_hobbies',  'property_damage']
ordinal = ['collision_type', 'incident_type', 'incident_severity']


**One Hot Encoder (OHE)**

In [ ]:
from sklearn.preprocessing import  OneHotEncoder , LabelEncoder , OrdinalEncoder

def OHE_cat(data , encoder_col = None , encoder = None) ->pd.DataFrame:

  data_ohe = data[nominal]

  if encoder == None:
    encoder = OneHotEncoder(
        handle_unknown="ignore" ,
        drop = "if_binary"
    )

    encoder.fit(data_ohe)
    encoder_col = encoder.get_feature_names_out(data_ohe.columns)


  data_encoded = encoder.transform(data_ohe).toarray()
  data_encoded = pd.DataFrame(data_encoded ,
                              index = data_ohe.index ,
                              columns = encoder_col
                              )


  return data_encoded , encoder_col , encoder

In [ ]:
df_OHE , encoder_col , encoder_type = OHE_cat(df)

**Oridinal Encoding**

In [ ]:
def OE_cat(data, encoder = None) -> pd.DataFrame:

    data_le = data[ordinal]

    collision_type = ['UNKNOWN', 'Side Collision', 'Rear Collision', 'Front Collision']
    incident_severity = ['Trivial Damage','Minor Damage','Major Damage','Total Loss']
    incident_type = ['Parked Car','Single Vehicle Collision','Multi-vehicle Collision','Vehicle Theft']

    if encoder == None:
        # Create object
        encoder = OrdinalEncoder(categories=[collision_type, incident_type,incident_severity])
        encoder.fit(data_le)

    ## Transform the data
    data_encoded = encoder.transform(data_le)
    data_encoded = pd.DataFrame(data_encoded,
                                index = data_le.index,
                                columns = data_le.columns)

    # save the object

    return data_encoded, encoder

In [ ]:
df_oe , encoder_type = OE_cat(df)

In [ ]:
# Concatenating categorical feature after applying OneHotEncoding , Oridinal Encoding
df = df.drop(columns=ordinal)
df = pd.concat([df, df_oe], axis=1)
df.shape

(2017, 38)

In [ ]:
df = df.drop(columns=nominal)
df = pd.concat([df, df_OHE], axis=1)
df.shape

(2017, 69)

In [ ]:
cat1_cols = []

for col in df.columns:
  if(df[col].dtypes == "object"):
    cat1_cols.append(col)


# **Label Encoding**

In [ ]:
from sklearn.preprocessing import  LabelEncoder

def label_encoding(data):

    le = LabelEncoder()

    data_encoded = le.fit_transform(data)

    return data_encoded



In [ ]:
df["insured_sex"] = label_encoding(df["insured_sex"])

In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:

  df[col] = label_encoding(df[col])


In [ ]:
df.to_csv("insurance_preprocessed_data.csv", index=False)
